In [1]:
import os

for root, dirs, files in os.walk("/home/sabrina/AI-Badminton-111"):
    if "merged_scores_imu.csv" in files:
        print(os.path.join(root, "merged_scores_imu.csv"))

/home/sabrina/AI-Badminton-111/merged_scores_imu.csv


In [2]:
import os
import re
import pandas as pd

folders = {
    "挑": "/home/sabrina/AI-Badminton-111/挑cut",
    "推": "/home/sabrina/AI-Badminton-111/推cut"
}

rows = []

for stroke_type, folder in folders.items():
    for f in os.listdir(folder):
        if not f.endswith(".csv"):
            continue

        m = re.match(r"^(\d+)(挑|推)(補)?_(\d+)\.csv$", f)

        if m:
            rows.append({
                "file_name": f,
                "subject_id": m.group(1),
                "stroke_type_x": stroke_type,
                "shot_id": int(m.group(4)),
                "path": os.path.join(folder, f)
            })

cut_df = pd.DataFrame(rows)

score_df = pd.read_csv("/home/sabrina/AI-Badminton-111/merged_scores_imu.csv")

score_cols = [
    "揮拍軌跡正確度",
    "揮拍速度流暢度",
    "手腕轉動時機正確度",
    "擊球時機正確度",
    "擊球位置正確度"
]

score_df["subject_id"] = score_df["file_name"].astype(str).str.extract(r"^(\d+)")

score_small = score_df[
    ["subject_id", "stroke_type_x"] + score_cols
].drop_duplicates()

merged_df = cut_df.merge(
    score_small,
    on=["subject_id", "stroke_type_x"],
    how="left"
)

merged_df = merged_df.dropna(subset=score_cols).reset_index(drop=True)

print("cut_df:", cut_df.shape)
print("merged_df:", merged_df.shape)
print("缺失值:")
print(merged_df[score_cols].isna().sum())
display(merged_df.head())

cut_df: (3488, 5)
merged_df: (3402, 10)
缺失值:
揮拍軌跡正確度      0
揮拍速度流暢度      0
手腕轉動時機正確度    0
擊球時機正確度      0
擊球位置正確度      0
dtype: int64


,file_name,subject_id,stroke_type_x,shot_id,path,揮拍軌跡正確度,揮拍速度流暢度,手腕轉動時機正確度,擊球時機正確度,擊球位置正確度
0,110117挑_7.csv,110117,挑,7,/home/sabrina/AI-Badminton-111/挑cut/110117挑_7.csv,1.0,2.0,1.0,3.0,2.0
1,110114挑_8.csv,110114,挑,8,/home/sabrina/AI-Badminton-111/挑cut/110114挑_8.csv,3.0,3.0,3.0,3.0,4.0
2,10305挑_13.csv,10305,挑,13,/home/sabrina/AI-Badminton-111/挑cut/10305挑_13.csv,2.0,5.0,3.0,5.0,1.0
3,110132挑_5.csv,110132,挑,5,/home/sabrina/AI-Badminton-111/挑cut/110132挑_5.csv,1.0,2.0,1.0,4.0,3.0
4,110131挑_7.csv,110131,挑,7,/home/sabrina/AI-Badminton-111/挑cut/110131挑_7.csv,3.0,4.0,2.0,4.0,5.0


In [3]:
# ↓↓↓ 這個 cell 是補回來的：建立 X（IMU 感測器陣列）跟 挑_idx / 推_idx
# 沒有這個 cell，下面訓練那格會找不到 X，直接報錯
import numpy as np

def load_imu_csv(csv_path):
    df = pd.read_csv(csv_path, header=None)
    df.columns = ["t1", "acc_x", "acc_y", "acc_z", "t2", "gyro_x", "gyro_y", "gyro_z"]
    return df[["acc_x", "acc_y", "acc_z", "gyro_x", "gyro_y", "gyro_z"]]

X_list = []
for _, row in merged_df.iterrows():
    imu_df = load_imu_csv(row["path"])
    X_list.append(imu_df.values)

X = np.array(X_list)
print("X shape:", X.shape)

挑_idx = merged_df["stroke_type_x"] == "挑"
推_idx = merged_df["stroke_type_x"] == "推"

print("挑球:", X[挑_idx].shape)
print("平推:", X[推_idx].shape)


X shape: (3402, 125, 6)
挑球: (1693, 125, 6)
平推: (1709, 125, 6)


In [6]:
def convert_score_to_category(score):
    score = int(score)
    return (score - 1) // 2   # 1,2 -> 0 ; 3,4 -> 1 ; 5 -> 2

score_cols = [
    "揮拍軌跡正確度",
    "揮拍速度流暢度",
    "手腕轉動時機正確度",
    "擊球時機正確度",
    "擊球位置正確度"
]

for col in score_cols:
    merged_df[col + "_cat"] = merged_df[col].apply(convert_score_to_category)

y1 = merged_df["揮拍軌跡正確度_cat"].values
y2 = merged_df["揮拍速度流暢度_cat"].values
y3 = merged_df["手腕轉動時機正確度_cat"].values
y4 = merged_df["擊球時機正確度_cat"].values
y5 = merged_df["擊球位置正確度_cat"].values

print("y1:", y1.shape)
print("y2:", y2.shape)
print("y3:", y3.shape)
print("y4:", y4.shape)
print("y5:", y5.shape)

y1: (3402,)
y2: (3402,)
y3: (3402,)
y4: (3402,)
y5: (3402,)


In [6]:
import numpy as np
import pandas as pd
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
import tensorflow as tf


from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import accuracy_score

np.random.seed(42)
tf.random.set_seed(42)

# =============================
# 先跑 model1
# =============================
metric_name = "揮拍軌跡正確度"
model_path = "專題/models/model1.keras"
y_all = y1

ratios = [0.1, 0.2, 0.3, 0.4, 0.5]
repeat_times = 3
epochs = 30

# =============================
# Scratch model
# =============================
def build_scratch_model(input_shape=(125, 6), num_classes=3):
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=input_shape),
        tf.keras.layers.LSTM(16),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(num_classes, activation="softmax")
    ])

    model.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model


# =============================
# Transfer model
# =============================
def build_transfer_model(base_model, num_classes=3):
    model = tf.keras.models.clone_model(base_model)
    model.set_weights(base_model.get_weights())

    model.pop()

    for layer in model.layers:
        layer.trainable = False

    model.add(tf.keras.layers.Dense(num_classes, activation="softmax"))

    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-3),  # 改成跟 scratch 一樣快，公平比較
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model


# =============================
# Load pretrained model
# =============================
model_path = "/home/sabrina/AI-Badminton-111/專題/models/model1.keras"
base_model_original = tf.keras.models.load_model(model_path)

all_results = []

for stroke_name, idx in [("挑球", 挑_idx), ("平推", 推_idx)]:

    print("正在跑:", metric_name, "-", stroke_name)

    X_target = X[idx]
    y_target = y_all[idx]

    target_df = merged_df[idx].reset_index(drop=True)
    groups = target_df["subject_id"]

    # 固定 train/test：以人為單位切
    splitter = GroupShuffleSplit(
        n_splits=1,
        test_size=0.2,
        random_state=42
    )

    train_idx, test_idx = next(
        splitter.split(X_target, y_target, groups=groups)
    )

    X_test = X_target[test_idx]
    y_test = y_target[test_idx]

    train_subjects = np.array(sorted(groups.iloc[train_idx].unique()))

    print("train subjects:", len(train_subjects))
    print("test subjects:", groups.iloc[test_idx].nunique())
    print("overlap:", set(groups.iloc[train_idx]) & set(groups.iloc[test_idx]))

    for run in range(repeat_times):

        seed = 42 + run
        rng = np.random.default_rng(seed)

        # 這裡固定一組受試者順序
        ordered_subjects = rng.permutation(train_subjects)

        for r in ratios:

            n_subjects = max(1, int(len(ordered_subjects) * r))
            selected_subjects = ordered_subjects[:n_subjects]

            small_idx = target_df[
                target_df["subject_id"].isin(selected_subjects)
            ].index.values

            X_small = X_target[small_idx]
            y_small = y_target[small_idx]

            print(
                f"{stroke_name} | run {run+1} | ratio {int(r*100)}% | "
                f"train samples: {len(X_small)} | "
                f"train subjects: {len(selected_subjects)}"
            )

            early_stop = tf.keras.callbacks.EarlyStopping(
                monitor="val_loss",   # 改成看驗證分數，不是只看訓練分數
                patience=5,
                restore_best_weights=True
            )

            # ---- Scratch ----
            tf.keras.backend.clear_session()

            scratch_model = build_scratch_model(
                input_shape=X_target.shape[1:]
            )

            scratch_model.fit(
                X_small,
                y_small,
                epochs=epochs,
                batch_size=8,
                verbose=0,
                validation_split=0.2,   # 切一部分訓練資料出來驗證，早停才有意義
                callbacks=[early_stop]
            )

            pred_scratch = scratch_model.predict(X_test, verbose=0)
            acc_scratch = accuracy_score(
                y_test,
                np.argmax(pred_scratch, axis=1)
            )

            # ---- Transfer ----
            tf.keras.backend.clear_session()

            transfer_model = build_transfer_model(base_model_original)

            transfer_model.fit(
                X_small,
                y_small,
                epochs=epochs,
                batch_size=8,
                verbose=0,
                validation_split=0.2,   # 切一部分訓練資料出來驗證，早停才有意義
                callbacks=[early_stop]
            )

            pred_transfer = transfer_model.predict(X_test, verbose=0)
            acc_transfer = accuracy_score(
                y_test,
                np.argmax(pred_transfer, axis=1)
            )

            all_results.append({
                "維度": metric_name,
                "動作": stroke_name,
                "run": run + 1,
                "資料比例": f"{int(r*100)}%",
                "train_subjects": len(selected_subjects),
                "train_samples": len(X_small),
                "Scratch_acc": acc_scratch,
                "Transfer_acc": acc_transfer
            })


raw_result_df = pd.DataFrame(all_results)
display(raw_result_df)

# 取平均與 std
result_df = raw_result_df.groupby(
    ["維度", "動作", "資料比例"],
    as_index=False
).agg(
    Scratch_mean=("Scratch_acc", "mean"),
    Scratch_std=("Scratch_acc", "std"),
    Transfer_mean=("Transfer_acc", "mean"),
    Transfer_std=("Transfer_acc", "std"),
    train_subjects=("train_subjects", "mean"),
    train_samples=("train_samples", "mean")
)

display(result_df)

# 存檔
result_df.to_csv(
    f"{metric_name}_累加受試者_finetune.csv",
    index=False,
    encoding="utf-8-sig"
)

raw_result_df.to_csv(
    f"{metric_name}_累加受試者_finetune_raw.csv",
    index=False,
    encoding="utf-8-sig"
)

正在跑: 揮拍軌跡正確度 - 挑球
train subjects: 47
test subjects: 12
overlap: set()
挑球 | run 1 | ratio 10% | train samples: 120 | train subjects: 4
挑球 | run 1 | ratio 20% | train samples: 254 | train subjects: 9
挑球 | run 1 | ratio 30% | train samples: 390 | train subjects: 14
挑球 | run 1 | ratio 40% | train samples: 497 | train subjects: 18
挑球 | run 1 | ratio 50% | train samples: 645 | train subjects: 23
挑球 | run 2 | ratio 10% | train samples: 112 | train subjects: 4
挑球 | run 2 | ratio 20% | train samples: 248 | train subjects: 9
挑球 | run 2 | ratio 30% | train samples: 400 | train subjects: 14
挑球 | run 2 | ratio 40% | train samples: 516 | train subjects: 18
挑球 | run 2 | ratio 50% | train samples: 668 | train subjects: 23
挑球 | run 3 | ratio 10% | train samples: 122 | train subjects: 4
挑球 | run 3 | ratio 20% | train samples: 254 | train subjects: 9
挑球 | run 3 | ratio 30% | train samples: 406 | train subjects: 14
挑球 | run 3 | ratio 40% | train samples: 522 | train subjects: 18
挑球 | run 3 | ratio 50% | t

,維度,動作,run,資料比例,train_subjects,train_samples,Scratch_acc,Transfer_acc
0,揮拍軌跡正確度,挑球,1,10%,4,120,0.235616,0.356164
1,揮拍軌跡正確度,挑球,1,20%,9,254,0.446575,0.323288
2,揮拍軌跡正確度,挑球,1,30%,14,390,0.454795,0.342466
3,揮拍軌跡正確度,挑球,1,40%,18,497,0.476712,0.342466
4,揮拍軌跡正確度,挑球,1,50%,23,645,0.219178,0.290411
5,揮拍軌跡正確度,挑球,2,10%,4,112,0.567123,0.246575
6,揮拍軌跡正確度,挑球,2,20%,9,248,0.339726,0.339726
7,揮拍軌跡正確度,挑球,2,30%,14,400,0.501370,0.391781
8,揮拍軌跡正確度,挑球,2,40%,18,516,0.468493,0.378082
9,揮拍軌跡正確度,挑球,2,50%,23,668,0.619178,0.408219


,維度,動作,資料比例,Scratch_mean,Scratch_std,Transfer_mean,Transfer_std,train_subjects,train_samples
0,揮拍軌跡正確度,平推,10%,0.486486,0.107145,0.433433,0.113692,4.0,121.333333
1,揮拍軌跡正確度,平推,20%,0.492492,0.069848,0.438438,0.143611,9.0,268.000000
2,揮拍軌跡正確度,平推,30%,0.576577,0.092510,0.377377,0.168418,13.0,384.666667
3,揮拍軌跡正確度,平推,40%,0.576577,0.044237,0.369369,0.195334,18.0,534.666667
4,揮拍軌跡正確度,平推,50%,0.584585,0.056208,0.433433,0.169725,23.0,685.000000
5,揮拍軌跡正確度,挑球,10%,0.379909,0.169870,0.320548,0.064077,4.0,118.000000
6,揮拍軌跡正確度,挑球,20%,0.386301,0.054726,0.286758,0.077942,9.0,252.000000
7,揮拍軌跡正確度,挑球,30%,0.492237,0.033815,0.320548,0.084355,14.0,398.666667
8,揮拍軌跡正確度,挑球,40%,0.497717,0.043693,0.357078,0.018649,18.0,511.666667
9,揮拍軌跡正確度,挑球,50%,0.427397,0.200506,0.335160,0.063803,23.0,657.000000


In [7]:
import numpy as np
import pandas as pd
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
import tensorflow as tf

from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import accuracy_score

np.random.seed(42)
tf.random.set_seed(42)

# =============================
# Scratch model
# =============================
def build_scratch_model(input_shape=(125, 6), num_classes=3):
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=input_shape),
        tf.keras.layers.LSTM(16),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(num_classes, activation="softmax")
    ])

    model.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model


# =============================
# Transfer model
# =============================
def build_transfer_model(base_model, num_classes=3):
    model = tf.keras.models.clone_model(base_model)
    model.set_weights(base_model.get_weights())

    model.pop()

    for layer in model.layers:
        layer.trainable = False

    model.add(tf.keras.layers.Dense(num_classes, activation="softmax"))

    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-3),  # 跟 scratch 一樣快，公平比較
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model


# =============================
# 四個要跑的指標
# =============================
metric_configs = [
    # {
    #     "metric_name": "揮拍速度流暢度",
    #     "model_path": "/home/sabrina/AI-Badminton-111/專題/models/model2.keras",
    #     "y_all": y2
    # },
    {
        "metric_name": "手腕轉動時機正確度",
        "model_path": "/home/sabrina/AI-Badminton-111/專題/models/model3.keras",
        "y_all": y3
    },
    {
        "metric_name": "擊球時機正確度",
        "model_path": "/home/sabrina/AI-Badminton-111/專題/models/model4.keras",
        "y_all": y4
    },
    {
        "metric_name": "擊球位置正確度",
        "model_path": "/home/sabrina/AI-Badminton-111/專題/models/model5.keras",
        "y_all": y5
    },
]

ratios = [0.1, 0.2, 0.3, 0.4, 0.5]
repeat_times = 3
epochs = 30

# =============================
# 依序跑 4 個指標
# =============================
for config in metric_configs:

    metric_name = config["metric_name"]
    model_path = config["model_path"]
    y_all = config["y_all"]

    print("=" * 50)
    print("開始跑指標:", metric_name)
    print("=" * 50)

    base_model_original = tf.keras.models.load_model(model_path)

    all_results = []

    for stroke_name, idx in [("挑球", 挑_idx), ("平推", 推_idx)]:

        print("正在跑:", metric_name, "-", stroke_name)

        X_target = X[idx]
        y_target = y_all[idx]

        target_df = merged_df[idx].reset_index(drop=True)
        groups = target_df["subject_id"]

        splitter = GroupShuffleSplit(
            n_splits=1,
            test_size=0.2,
            random_state=42
        )

        train_idx, test_idx = next(
            splitter.split(X_target, y_target, groups=groups)
        )

        X_test = X_target[test_idx]
        y_test = y_target[test_idx]

        train_subjects = np.array(sorted(groups.iloc[train_idx].unique()))

        print("train subjects:", len(train_subjects))
        print("test subjects:", groups.iloc[test_idx].nunique())
        print("overlap:", set(groups.iloc[train_idx]) & set(groups.iloc[test_idx]))

        for run in range(repeat_times):

            seed = 42 + run
            rng = np.random.default_rng(seed)

            ordered_subjects = rng.permutation(train_subjects)

            for r in ratios:

                n_subjects = max(1, int(len(ordered_subjects) * r))
                selected_subjects = ordered_subjects[:n_subjects]

                small_idx = target_df[
                    target_df["subject_id"].isin(selected_subjects)
                ].index.values

                X_small = X_target[small_idx]
                y_small = y_target[small_idx]

                print(
                    f"{stroke_name} | run {run+1} | ratio {int(r*100)}% | "
                    f"train samples: {len(X_small)} | "
                    f"train subjects: {len(selected_subjects)}"
                )

                early_stop = tf.keras.callbacks.EarlyStopping(
                    monitor="val_loss",
                    patience=5,
                    restore_best_weights=True
                )

                # ---- Scratch ----
                tf.keras.backend.clear_session()

                scratch_model = build_scratch_model(
                    input_shape=X_target.shape[1:]
                )

                scratch_model.fit(
                    X_small,
                    y_small,
                    epochs=epochs,
                    batch_size=8,
                    verbose=0,
                    validation_split=0.2,
                    callbacks=[early_stop]
                )

                pred_scratch = scratch_model.predict(X_test, verbose=0)
                acc_scratch = accuracy_score(
                    y_test,
                    np.argmax(pred_scratch, axis=1)
                )

                # ---- Transfer ----
                tf.keras.backend.clear_session()

                transfer_model = build_transfer_model(base_model_original)

                transfer_model.fit(
                    X_small,
                    y_small,
                    epochs=epochs,
                    batch_size=8,
                    verbose=0,
                    validation_split=0.2,
                    callbacks=[early_stop]
                )

                pred_transfer = transfer_model.predict(X_test, verbose=0)
                acc_transfer = accuracy_score(
                    y_test,
                    np.argmax(pred_transfer, axis=1)
                )

                all_results.append({
                    "維度": metric_name,
                    "動作": stroke_name,
                    "run": run + 1,
                    "資料比例": f"{int(r*100)}%",
                    "train_subjects": len(selected_subjects),
                    "train_samples": len(X_small),
                    "Scratch_acc": acc_scratch,
                    "Transfer_acc": acc_transfer
                })

    raw_result_df = pd.DataFrame(all_results)
    display(raw_result_df)

    result_df = raw_result_df.groupby(
        ["維度", "動作", "資料比例"],
        as_index=False
    ).agg(
        Scratch_mean=("Scratch_acc", "mean"),
        Scratch_std=("Scratch_acc", "std"),
        Transfer_mean=("Transfer_acc", "mean"),
        Transfer_std=("Transfer_acc", "std"),
        train_subjects=("train_subjects", "mean"),
        train_samples=("train_samples", "mean")
    )

    display(result_df)

    result_df.to_csv(
        f"{metric_name}_累加受試者_finetune.csv",
        index=False,
        encoding="utf-8-sig"
    )

    raw_result_df.to_csv(
        f"{metric_name}_累加受試者_finetune_raw.csv",
        index=False,
        encoding="utf-8-sig"
    )

    print("完成:", metric_name, "\n")

print("四個指標全部跑完！")

I0000 00:00:1786582697.059864  445969 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1786582697.063401  445969 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1786582697.523640  445969 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1786582701.690893  445969 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:0

開始跑指標: 手腕轉動時機正確度


E0000 00:00:1786582704.146262  445969 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


正在跑: 手腕轉動時機正確度 - 挑球
train subjects: 47
test subjects: 12
overlap: set()
挑球 | run 1 | ratio 10% | train samples: 120 | train subjects: 4
挑球 | run 1 | ratio 20% | train samples: 254 | train subjects: 9
挑球 | run 1 | ratio 30% | train samples: 390 | train subjects: 14
挑球 | run 1 | ratio 40% | train samples: 497 | train subjects: 18
挑球 | run 1 | ratio 50% | train samples: 645 | train subjects: 23
挑球 | run 2 | ratio 10% | train samples: 112 | train subjects: 4
挑球 | run 2 | ratio 20% | train samples: 248 | train subjects: 9
挑球 | run 2 | ratio 30% | train samples: 400 | train subjects: 14
挑球 | run 2 | ratio 40% | train samples: 516 | train subjects: 18
挑球 | run 2 | ratio 50% | train samples: 668 | train subjects: 23
挑球 | run 3 | ratio 10% | train samples: 122 | train subjects: 4
挑球 | run 3 | ratio 20% | train samples: 254 | train subjects: 9
挑球 | run 3 | ratio 30% | train samples: 406 | train subjects: 14
挑球 | run 3 | ratio 40% | train samples: 522 | train subjects: 18
挑球 | run 3 | ratio 50% |

,維度,動作,run,資料比例,train_subjects,train_samples,Scratch_acc,Transfer_acc
0,手腕轉動時機正確度,挑球,1,10%,4,120,0.345205,0.397260
1,手腕轉動時機正確度,挑球,1,20%,9,254,0.391781,0.364384
2,手腕轉動時機正確度,挑球,1,30%,14,390,0.484932,0.465753
3,手腕轉動時機正確度,挑球,1,40%,18,497,0.465753,0.238356
4,手腕轉動時機正確度,挑球,1,50%,23,645,0.287671,0.391781
5,手腕轉動時機正確度,挑球,2,10%,4,112,0.419178,0.386301
6,手腕轉動時機正確度,挑球,2,20%,9,248,0.287671,0.205479
7,手腕轉動時機正確度,挑球,2,30%,14,400,0.336986,0.312329
8,手腕轉動時機正確度,挑球,2,40%,18,516,0.295890,0.353425
9,手腕轉動時機正確度,挑球,2,50%,23,668,0.375342,0.339726


...

,維度,動作,資料比例,Scratch_mean,Scratch_std,Transfer_mean,Transfer_std,train_subjects,train_samples
0,擊球時機正確度,平推,10%,0.367367,0.036697,0.281281,0.180088,4.0,121.333333
1,擊球時機正確度,平推,20%,0.517518,0.108994,0.349349,0.084423,9.0,268.000000
2,擊球時機正確度,平推,30%,0.482482,0.143590,0.404404,0.125708,13.0,384.666667
3,擊球時機正確度,平推,40%,0.587588,0.057162,0.265265,0.124410,18.0,534.666667
4,擊球時機正確度,平推,50%,0.503504,0.102161,0.360360,0.161605,23.0,685.000000
5,擊球時機正確度,挑球,10%,0.326941,0.046035,0.332420,0.038971,4.0,118.000000
6,擊球時機正確度,挑球,20%,0.382648,0.139224,0.347945,0.115068,9.0,252.000000
7,擊球時機正確度,挑球,30%,0.402740,0.096008,0.478539,0.043693,14.0,398.666667
8,擊球時機正確度,挑球,40%,0.457534,0.038356,0.333333,0.106685,18.0,511.666667
9,擊球時機正確度,挑球,50%,0.348858,0.037798,0.390868,0.127694,23.0,657.000000


完成: 擊球時機正確度 

開始跑指標: 擊球位置正確度
正在跑: 擊球位置正確度 - 挑球
train subjects: 47
test subjects: 12
overlap: set()
挑球 | run 1 | ratio 10% | train samples: 120 | train subjects: 4
挑球 | run 1 | ratio 20% | train samples: 254 | train subjects: 9
挑球 | run 1 | ratio 30% | train samples: 390 | train subjects: 14
挑球 | run 1 | ratio 40% | train samples: 497 | train subjects: 18
挑球 | run 1 | ratio 50% | train samples: 645 | train subjects: 23
挑球 | run 2 | ratio 10% | train samples: 112 | train subjects: 4
挑球 | run 2 | ratio 20% | train samples: 248 | train subjects: 9
挑球 | run 2 | ratio 30% | train samples: 400 | train subjects: 14
挑球 | run 2 | ratio 40% | train samples: 516 | train subjects: 18
挑球 | run 2 | ratio 50% | train samples: 668 | train subjects: 23
挑球 | run 3 | ratio 10% | train samples: 122 | train subjects: 4
挑球 | run 3 | ratio 20% | train samples: 254 | train subjects: 9
挑球 | run 3 | ratio 30% | train samples: 406 | train subjects: 14
挑球 | run 3 | ratio 40% | train samples: 522 | train subjects: 

,維度,動作,run,資料比例,train_subjects,train_samples,Scratch_acc,Transfer_acc
0,擊球位置正確度,挑球,1,10%,4,120,0.186301,0.408219
1,擊球位置正確度,挑球,1,20%,9,254,0.375342,0.342466
2,擊球位置正確度,挑球,1,30%,14,390,0.265753,0.353425
3,擊球位置正確度,挑球,1,40%,18,497,0.328767,0.367123
4,擊球位置正確度,挑球,1,50%,23,645,0.416438,0.257534
5,擊球位置正確度,挑球,2,10%,4,112,0.397260,0.386301
6,擊球位置正確度,挑球,2,20%,9,248,0.471233,0.375342
7,擊球位置正確度,挑球,2,30%,14,400,0.350685,0.432877
8,擊球位置正確度,挑球,2,40%,18,516,0.430137,0.331507
9,擊球位置正確度,挑球,2,50%,23,668,0.484932,0.350685


,維度,動作,資料比例,Scratch_mean,Scratch_std,Transfer_mean,Transfer_std,train_subjects,train_samples
0,擊球位置正確度,平推,10%,0.512513,0.226330,0.468468,0.074351,4.0,121.333333
1,擊球位置正確度,平推,20%,0.495495,0.006006,0.208208,0.124193,9.0,268.000000
2,擊球位置正確度,平推,30%,0.583584,0.007557,0.440440,0.090953,13.0,384.666667
3,擊球位置正確度,平推,40%,0.532533,0.035827,0.474474,0.077673,18.0,534.666667
4,擊球位置正確度,平推,50%,0.507508,0.150030,0.434434,0.143935,23.0,685.000000
5,擊球位置正確度,挑球,10%,0.292237,0.105482,0.420091,0.041035,4.0,118.000000
6,擊球位置正確度,挑球,20%,0.389041,0.076271,0.357991,0.016514,9.0,252.000000
7,擊球位置正確度,挑球,30%,0.363470,0.104697,0.334247,0.109486,14.0,398.666667
8,擊球位置正確度,挑球,40%,0.368037,0.054405,0.303196,0.081841,18.0,511.666667
9,擊球位置正確度,挑球,50%,0.432877,0.046089,0.288584,0.053781,23.0,657.000000


完成: 擊球位置正確度 

四個指標全部跑完！
